In [0]:
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

In [0]:
# Load active freshness validation configurations
registry_table = "validation_sweep.metadata.freshness_registry"
results_table = "validation_sweep.results.validation_results"

registrations = (
    spark.table(registry_table)
    .filter(F.col("is_active") == True)
    .select(
        "catalog_name",
        "schema_name",
        "table_name",
        "max_age_hours",
    )
    .collect()
)

print(f"Active freshness checks: {len(registrations)}")

for row in registrations:
    print(
        f"{row.catalog_name}.{row.schema_name}.{row.table_name} "
        f"(max age: {row.max_age_hours} hours)"
    )

In [0]:
# Define Delta operations that qualify as data changes
# Exclude OPTIMIZE because it reorganizes files without changing table data
data_change_operations = [
    "WRITE",
    "CREATE TABLE AS SELECT",
    "MERGE",
    "UPDATE",
    "DELETE",
    "COPY INTO",
    "STREAMING UPDATE",
]

In [0]:
# Calculate freshness with per-table error isolation
results = []
run_ts = datetime.now(timezone.utc)

for row in registrations:
    table_name = f"{row.catalog_name}.{row.schema_name}.{row.table_name}"

    result = {
        "table_name": table_name,
        "last_data_change_ts": None,
        "age_hours": None,
        "max_age_hours": row.max_age_hours,
        "test_result": "ERROR",
        "error_message": None,
    }

    try:
        history_df = spark.sql(f"describe history {table_name}")

        latest_change = (
            history_df
            .filter(F.col("operation").isin(data_change_operations))
            .orderBy(F.col("timestamp").desc())
            .select("timestamp", "operation")
            .first()
        )

        if latest_change is None:
            result["error_message"] = "No identifiable data-changing operation"
        else:
            last_data_change_ts = latest_change.timestamp.replace(tzinfo=timezone.utc)
            age_hours = (run_ts - last_data_change_ts).total_seconds() / 3600

            result["last_data_change_ts"] = latest_change.timestamp
            result["age_hours"] = age_hours
            result["test_result"] = (
                "PASS" if age_hours <= row.max_age_hours else "FAIL"
            )

    except Exception as e:
        result["error_message"] = str(e).strip().splitlines()[0][:1000]

    results.append(result)

    print(
        f"{table_name}: {result['test_result']} "
        f"(age: {result['age_hours']}, "
        f"threshold: {result['max_age_hours']} hours)"
    )

In [0]:
# Persist freshness validation results to the central Delta table
run_id = str(uuid4())

method = "python_native"
test_type = "freshness"

records = []

for result in results:
    records.append({
        "run_id": run_id,
        "run_ts": run_ts.replace(tzinfo=None),
        "run_date": run_ts.date(),
        "check_id": "|".join([
            method,
            test_type,
            result["table_name"],
        ]),
        "method": method,
        "test_type": test_type,
        "table_name": result["table_name"],
        "key_columns": None,
        "test_result": result["test_result"],
        "last_data_change_ts": result["last_data_change_ts"],
        "age_hours": result["age_hours"],
        "max_age_hours": result["max_age_hours"],
        "error_message": result["error_message"],
    })

target_schema = spark.table(results_table).schema
column_names = target_schema.fieldNames()

rows = [
    tuple(record.get(column) for column in column_names)
    for record in records
]

if rows:
    results_df = spark.createDataFrame(rows, schema=target_schema)

    (
        results_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(results_table)
    )

    print(f"Persisted {len(rows)} freshness results")
    print(f"Run ID: {run_id}")
else:
    print("No active freshness checks to persist")

In [0]:
%skip
%sql
-- Inspect historical freshness error-isolation test results
select
    run_id,
    run_ts,
    method,
    test_type,
    table_name,
    test_result,
    last_data_change_ts,
    round(age_hours, 2) as age_hours,
    max_age_hours,
    error_message
from validation_sweep.results.validation_results
where run_id = '857b799d-29bf-42f8-ad4b-0a16134cb361'
order by table_name;

In [0]:
%skip
%sql
-- Verify freshness results appear in the latest-results view
select
    run_id,
    run_ts,
    method,
    test_type,
    table_name,
    test_result,
    round(age_hours, 2) as age_hours,
    max_age_hours
from validation_sweep.results.latest_validation_results
where method = 'python_native'
    and test_type = 'freshness'
order by table_name;